# Day 2 | Notebook 0: RAG Foundations
**Author: Sattaya Singkul**

> 📖 This is a **theory notebook** — no code to run. Read it carefully before starting NB01.
> It serves as your reference sheet for the entire Day 2 workshop.

---

## 1. What is RAG and Why Does It Exist?

### The Core Problem

Large Language Models (LLMs) like GPT-4 or Llama are trained on public internet data.
They have **two critical limitations** for enterprise use:

1. **No private knowledge** — They don't know your product catalog, internal docs, or customer data.
2. **Hallucination** — They generate confident-sounding answers that are factually wrong.

**RAG (Retrieval-Augmented Generation)** solves both:

```
User Question
     │
     ▼
[Retrieve] ─── Search Vector DB for relevant facts
     │
     ▼
[Augment]  ─── Inject facts into LLM prompt
     │
     ▼
[Generate] ─── LLM answers ONLY from provided facts
     │
     ▼
 Grounded Answer (verifiable, no hallucination)
```

### The RAG Taxonomy

| Type | Description | When to use |
|---|---|---|
| **Naive RAG** | Chunk → Embed → Query → Generate | Quick prototypes |
| **Advanced RAG** | Better chunking + reranking + guardrails | Production systems |
| **Modular RAG** | Pluggable retrievers, readers, generators | Enterprise / research |

## 2. The RAG Pipeline — 6 Stages

```
╔══════════════════════════════════════════════════════════════╗
║  OFFLINE (Ingestion)              ONLINE (Query)             ║
║                                                              ║
║  Raw Docs                         User Question              ║
║     │                                  │                     ║
║  [Stage 1]                         [Stage 2]                 ║
║  Chunking                          Embed Query               ║
║     │                                  │                     ║
║  [Stage 1]                         [Stage 2]                 ║
║  Embedding                         ANN Search                ║
║     │                                  │                     ║
║  [Stage 1]                         [Stage 3]                 ║
║  Store in                          Reranking                 ║
║  Redis                             (optional)                ║
║                                        │                     ║
║                                   [Stage 4]                  ║
║                                   Augmentation               ║
║                                   (build prompt)             ║
║                                        │                     ║
║                                   [Stage 5]                  ║
║                                   Generation                 ║
║                                   (LLM answer)               ║
║                                        │                     ║
║                                   [Stage 6]                  ║
║                                   Evaluation                 ║
║                                   (score + guard)            ║
╚══════════════════════════════════════════════════════════════╝
```

**In this workshop, RedisVL handles Stages 1, 2, 3, and parts of 6.**

## 3. Chunking Strategies

How you split text dramatically affects retrieval quality.

### Fixed-Size Chunking
```
Text:   [----chunk1----][----chunk2----][----chunk3----]
Pros:   Simple, fast
Cons:   Breaks sentences mid-thought, loses context
Use:    Raw text where sentence boundaries don't matter
```

### Sentence Chunking
```
Text:   [Sentence 1.] [Sentence 2.] [Sentence 3.]
Pros:   Natural language units
Cons:   Variable size, sentences can be uninformative alone
Use:    General documents
```

### Propositional / Atomic Chunking (⭐ Used in this workshop)
```
Input:  'The Aether X1 has RTX 4090 and costs $2,499.'
Output: ['Aether X1 has RTX 4090',
         'Aether X1 costs $2,499']

Pros:   Maximum precision — each fact is searchable independently
Cons:   More index entries
Use:    Product catalogs, FAQs, structured knowledge bases
```

### Parent-Document Retrieval (PDR) Pattern
```
Index:   [child fact 1] ──► parent_id='doc_A'
         [child fact 2] ──► parent_id='doc_A'
         [child fact 3] ──► parent_id='doc_B'

Query:   Search child facts (small → precise)
Return:  Fetch full parent document (large → rich context)

Result:  High precision retrieval + full context for generation
```

## 4. RAG Failure Modes

Understanding failures is as important as building the pipeline.

| Failure Mode | Symptom | Root Cause | Fix |
|---|---|---|---|
| **Hallucination** | Confident but wrong answer | LLM adds facts not in context | Faithfulness guard |
| **Retrieval Failure** | Relevant docs not returned | Poor chunking, wrong threshold | Tune threshold, better chunks |
| **Context Mismatch** | Right docs, wrong answer | Ambiguous or noisy context | Reranking, better prompting |
| **Lost in the Middle** | Long context, LLM ignores middle | Attention bias in transformers | Reduce retrieved docs, reorder |
| **Stale Knowledge** | Outdated answer | Vector DB not refreshed after data change | Scheduled re-ingestion, TTL |

## 5. Evaluation Framework (RAGAS Concepts)

You need numbers, not feelings, to know if your RAG is working.

### The 4 Core Metrics

#### Faithfulness
```
Faithfulness = (claims in answer that are in context) / (total claims in answer)

Answer: 'GPU uses liquid cooling and costs $2,000'
Context: 'GPU uses liquid cooling. Price is $2,499.'

Claim 'liquid cooling' ✓ in context
Claim '$2,000'        ✗ not in context (hallucination!)

Faithfulness = 1/2 = 0.5 → below threshold → REFUSAL
```

#### Answer Relevancy
```
Relevancy = cosine_similarity(embed(question), embed(answer))
Range: 0.0 (irrelevant) → 1.0 (perfectly on-topic)
```

#### Context Recall
```
Recall = (ground_truth facts found in retrieved context) / (total ground_truth facts)
Low recall → you're not retrieving enough relevant docs
```

#### Context Precision
```
Precision = (useful retrieved docs) / (total retrieved docs)
Low precision → too much noise in retrieved context
```

### The Tradeoff
```
Strict distance_threshold → high precision, low recall
Loose distance_threshold  → high recall, low precision

Goal: find the threshold where BOTH are acceptable for your use case.
```

## 6. RedisVL Architecture — What We're Building

```
┌─────────────────────────────────────────────────────────────────┐
│                     Redis Stack (Port 6379)                      │
│                                                                  │
│  ┌──────────────┐  ┌──────────────┐  ┌────────────────────────┐ │
│  │  PDR Index   │  │ Intent Index │  │   Semantic Cache       │ │
│  │  pdr_knowledge│  │ intent_router│  │   rag_cache            │ │
│  │  (propositions│  │ (10 examples)│  │   (Q→A pairs)          │ │
│  │  + parent_id)│  │              │  │                        │ │
│  └──────────────┘  └──────────────┘  └────────────────────────┘ │
│                                                                  │
│  ┌──────────────────────────────────────────────────────────┐   │
│  │              SemanticMessageHistory                       │   │
│  │              (per-user conversation turns)                │   │
│  └──────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────┘
              ▲                      ▲
              │                      │
   ┌──────────────────┐   ┌──────────────────────┐
   │   FastAPI        │   │   Jupyter Notebooks   │
   │   Port 8000      │   │   Port 8888           │
   │   (app.py)       │   │   (NB01 – NB06)       │
   └──────────────────┘   └──────────────────────┘
```

### RedisVL Classes We Use

| Class | Purpose | Notebook |
|---|---|---|
| `AsyncSearchIndex` | Index CRUD + async load/query | NB01, NB04 |
| `SearchIndex` | Sync version for simpler demos | NB02, NB03 |
| `VectorQuery` | KNN vector similarity search | NB02, NB04 |
| `RangeQuery` | Radius-based similarity search | NB02 |
| `HFTextVectorizer` | Local HuggingFace embedding model | NB04, NB05 |
| `SemanticCache` | LLM response caching | NB04, NB06 |
| `SemanticMessageHistory` | Conversation memory | NB03, NB06 |
| `Tag`, `Num`, `Text` filter | Metadata pre-filtering | NB02 |

## 7. Production Checklist

Before shipping a RAG system, verify:

```
Ingestion
  ☐ Chunking strategy validated on representative queries
  ☐ Embedding model chosen for your domain
  ☐ Bulk async ingestion tested (throughput benchmarked)

Retrieval
  ☐ distance_threshold tuned (plot Precision@K vs Recall@K)
  ☐ Hybrid search (semantic + keyword) for exact-match queries
  ☐ Intent routing prevents cross-domain hallucination

Generation
  ☐ Faithfulness guard blocking score < 0.6
  ☐ Semantic cache reduces LLM load (target >40% hit rate)
  ☐ Context window budget: prompt + context + answer fits in limit

Operations
  ☐ /health endpoint for Kubernetes readiness probe
  ☐ /stats endpoint for monitoring dashboards
  ☐ Cache TTL configured to avoid stale answers
  ☐ Agent memory TTL for privacy compliance (PDPA/GDPR)
  ☐ Load test: p95 latency < 2s at target RPS
```

---

> ✅ **You're ready to start NB01.** Keep this notebook open as a reference.